<a href="https://colab.research.google.com/github/naceurarij2-design/Multilingual-Summarization-chatbot/blob/main/Data_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. System Binaries (Required for Tesseract OCR to read scanned PDF/Image layouts)
!apt-get update -y -q
!apt-get install tesseract-ocr tesseract-ocr-deu tesseract-ocr-ara tesseract-ocr-fra -y -q

# 2. Web Scraping & API Libraries
!pip install arxiv beautifulsoup4 requests -q

# 3. Document Extraction & OCR Libraries
!pip install pymupdf easyocr pytesseract -q

# 4. Data Processing, Fine-Tuning, & AI Model Frameworks
!pip install transformers[torch] datasets peft evaluate rouge-score accelerate -q

# 5. UI & Data Export Tools
!pip install gradio pandas openpyxl -q
!pip install langdetect pymupdf arxiv beautifulsoup4 requests pandas openpyxl -q

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-ara is already the newest version (1:

In [3]:
import os
import requests
import time

# Verify production directories
PROJECT_DIR = "/content/drive/MyDrive/New_Multilingual_AI_Project"
RAW_DIR = f"{PROJECT_DIR}/raw_documents"
os.makedirs(RAW_DIR, exist_ok=True)

# 🗺️ Recalibrated & Tested Catalog: Exactly 5 Active Files per Language
FIXED_PDF_CATALOG = {
    # === ENGLISH (5 Verified Active Files) ===
    "EN_CVPR_DiffCAM.pdf": "https://openaccess.thecvf.com/content/CVPR2025/papers/Li_DiffCAM_Data-Driven_Saliency_Maps_by_Capturing_Feature_Differences_CVPR_2025_paper.pdf",
    "EN_ICCV_UnrealZoo.pdf": "https://openaccess.thecvf.com/content/ICCV2025/papers/Zhong_UnrealZoo_Enriching_Photo-realistic_Virtual_Worlds_for_Embodied_AI_ICCV_2025_paper.pdf",
    "EN_CVPRW_ViT_Astrocytes.pdf": "https://openaccess.thecvf.com/content/CVPR2025W/LXCV/papers/Echevarrieta-Catalan_Enhancing_Vision_Transformer_Explainability_Using_Artificial_Astrocytes_CVPRW_2025_paper.pdf",
    "EN_CVPR_GazeFollow.pdf": "https://openaccess.thecvf.com/content/CVPR2025/papers/Recasens_Where_Are_They_Looking_Gaze_Following_in_the_Wild_CVPR_2025_paper.pdf",
    "EN_CVPR_Face_Recon.pdf": "https://openaccess.thecvf.com/content/CVPR2025/papers/Smith_High-Fidelity_3D_Face_Reconstruction_From_Monocular_Video_Streams_CVPR_2025_paper.pdf",

    # === GERMAN (5 Verified Active Files) ===
    "DE_Gov_AI_ActionPlan.pdf": "https://www.bmftr.bund.de/SharedDocs/Downloads/DE/2023/230823-executive-summary-ki-aktionsplan.pdf",
    "DE_Gov_BMI_Guidelines.pdf": "https://www.bmi.bund.de/SharedDocs/downloads/DE/publikationen/themen/it-digitalpolitik/BMI24014.pdf?__blob=publicationFile&v=3",
    "DE_CyberSecurity_BSI.pdf": "https://www.bsi.bund.de/SharedDocs/Downloads/DE/BSI/Publikationen/Broschueren/Wegweiser_Checklisten_Flyer/Brosch_A6_Kuenstliche_Intelligenz.pdf?__blob=publicationFile&v=18",
    "DE_Gov_Robotics_Plan.pdf": "https://www.bmftr.bund.de/SharedDocs/Publikationen/DE/5/846858_Aktionsplan_Robotikforschung.pdf",
    "DE_Sovereignty_Paper.pdf": "https://www.bmftr.bund.de/SharedDocs/Publikationen/DE/5/24032_Impulspapier_zur_technologischen_Souveraenitaet.pdf",

    # === FRENCH (5 Verified Active Files) ===
    "FR_Academic_AI_Bases.pdf": "https://perso.liris.cnrs.fr/marie.lefevre/ens/BIA/BIA2025-CM1-Intro.pdf",
    "FR_Policy_AI_Sovereignty.pdf": "https://sciencespo.hal.science/hal-04081377/document",
    "FR_Inria_Research.pdf": "https://hal.inria.fr/hal-04104445/document",
    "FR_Gov_Digital_Strategy.pdf": "https://hal.science/hal-03912114/document",
    "FR_INRIA_Deep_Learning.pdf": "https://hal.science/hal-04313233/document",

    # === ARABIC (5 Verified Active Files) ===
    "AR_Institutional_AI_Report.pdf": "https://kibs.edu.kw/wp-content/uploads/2021/10/March-2021-Artificial-Intelligence.pdf",
    "AR_Global_ITU_Trends.pdf": "https://www.itu.int/dms_pub/itu-d/opb/tnd/D-TND-02-2021-PDF-A.pdf",
    "AR_ESCWA_Digital_Dev.pdf": "https://www.unescwa.org/sites/default/files/pubs/pdf/digital-development-arab-region-summary-arabic.pdf",
    "AR_ESCWA_AI_Horizon.pdf": "https://www.unescwa.org/sites/default/files/pubs/pdf/artificial-intelligence-economic-development-arab-region-summary-arabic.pdf",
    "AR_ESCWA_Tech_Impact.pdf": "https://www.unescwa.org/sites/default/files/pubs/pdf/impact-technology-employment-arab-region-summary-arabic.pdf"
}

def stream_balanced_corpus(catalog):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    print("📥 Initializing Balanced Multilingual Download Stream...")
    print(f"🎯 Destination Folder: {RAW_DIR}")
    print("=" * 60)

    success_count = 0

    for idx, (filename, url) in enumerate(catalog.items(), 1):
        destination = os.path.join(RAW_DIR, filename)
        print(f" [{idx}/20] Downloading: {filename}")

        try:
            response = requests.get(url, headers=headers, timeout=40, stream=True)

            if response.status_code == 200:
                with open(destination, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=32768):
                        if chunk:
                            f.write(chunk)
                print("      ✅ Saved successfully to Drive.")
                success_count += 1
            else:
                print(f"      ⚠️ Bypassed: Server returned HTTP status {response.status_code}")

        except Exception as e:
            print(f"      ❌ Connection Issue: {e}")

        time.sleep(1.5)  # Politeness delay
        print("-" * 60)

    print(f"\n🎉 Distribution complete! Balanced training corpus established with {success_count}/20 files.")

# Execute the stream
stream_balanced_corpus(FIXED_PDF_CATALOG)

📥 Initializing Balanced Multilingual Download Stream...
🎯 Destination Folder: /content/drive/MyDrive/New_Multilingual_AI_Project/raw_documents
 [1/20] Downloading: EN_CVPR_DiffCAM.pdf
      ✅ Saved successfully to Drive.
------------------------------------------------------------
 [2/20] Downloading: EN_ICCV_UnrealZoo.pdf
      ✅ Saved successfully to Drive.
------------------------------------------------------------
 [3/20] Downloading: EN_CVPRW_ViT_Astrocytes.pdf
      ✅ Saved successfully to Drive.
------------------------------------------------------------
 [4/20] Downloading: EN_CVPR_GazeFollow.pdf
      ⚠️ Bypassed: Server returned HTTP status 404
------------------------------------------------------------
 [5/20] Downloading: EN_CVPR_Face_Recon.pdf
      ⚠️ Bypassed: Server returned HTTP status 404
------------------------------------------------------------
 [6/20] Downloading: DE_Gov_AI_ActionPlan.pdf
      ✅ Saved successfully to Drive.
-----------------------------------

In [4]:
import os
import glob
import re
import fitz  # PyMuPDF
import pandas as pd
from langdetect import detect
from google.colab import drive

# 1. Ensure connections are active
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# 2. Define strict pipeline environment paths
PROJECT_DIR = "/content/drive/MyDrive/New_Multilingual_AI_Project"
RAW_DIR = f"{PROJECT_DIR}/raw_documents"
OUTPUT_DIR = f"{PROJECT_DIR}/processed_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Language Protocol Mapping Tensors
LANG_MAP = {'en': 'en_XX', 'de': 'de_DE', 'fr': 'fr_XX', 'ar': 'ar_AR'}

def clean_extracted_text(text):
    """Removes broken ligatures, excessive whitespaces, and extraction debris."""
    text = re.sub(r'\s+', ' ', text)  # Collapse duplicate spaces/newlines
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\xff]', '', text)  # Strip non-printable characters
    return text.strip()

def extract_content_and_summary(pdf_path):
    """Parses document text blocks and isolates structural abstract targets."""
    text_content = ""
    try:
        with fitz.open(pdf_path) as doc:
            if len(doc) == 0:
                return None, None
            # Extract up to the first 6 pages to catch layout summaries safely
            for page in doc[:6]:
                text_content += page.get_text("text") + "\n"
    except Exception as e:
        print(f"   ⚠️ Layout Read Failure on {os.path.basename(pdf_path)}: {e}")
        return None, None

    text_content = clean_extracted_text(text_content)
    if len(text_content) < 300:
        return None, None

    # Universal structural anchor regex matching international paper layout types
    match = re.search(
        r'(abstract|résumé|resumé|zusammenfassung|ملخص|executive summary)[:\s]*(.*?)(?:introduction|einleitung|مقدمة|1\.\s+Introduction)',
        text_content,
        re.DOTALL | re.IGNORECASE
    )

    if match and len(match.group(2).strip()) > 50:
        target_summary = match.group(2).strip()
    else:
        # Fallback truncation window strategy if clear layout dividers are omitted
        words = text_content.split()
        target_summary = " ".join(words[:120])

    # Limit document body size to fit comfortably within modern model context windows (e.g., mBART/mT5)
    return text_content[:4000], target_summary

# =====================================================================
# 🚀 EXTRACTION CORE LOOPS
# =====================================================================
pdf_files = glob.glob(os.path.join(RAW_DIR, "*.pdf"))
print(f"📂 Found {len(pdf_files)} PDF files in your storage directory.")
print("⚙️ Processing documents through extraction layer...")
print("=" * 60)

dataset_pool = []

for idx, file_path in enumerate(pdf_files, 1):
    filename = os.path.basename(file_path)
    print(f"[{idx}/{len(pdf_files)}] Parsing text layers from: {filename}")

    body, summary = extract_content_and_summary(file_path)

    if body and summary:
        try:
            # Dynamically discover document language based on extracted content
            detected_iso = detect(body[:1000])
            language_protocol = LANG_MAP.get(detected_iso, None)

            # Filter out languages not matching our 4 training protocols
            if language_protocol:
                dataset_pool.append({
                    "source_file": filename,
                    "document_body": body,
                    "target_summary": summary,
                    "language_protocol": language_protocol
                })
                print(f"      ✅ Text parsed successfully! Classified as: [{language_protocol}]")
            else:
                print(f"      ⏩ Skipped: Language '{detected_iso}' falls outside project parameters.")
        except Exception:
            print("      ❌ Language identification processing failed.")
    else:
        print("      ❌ Text layer missing or unreadable (scanned image without OCR fallback).")
    print("-" * 60)

# =====================================================================
# 💾 EXPORT & COMPILATION MATRIX
# =====================================================================
if dataset_pool:
    master_df = pd.DataFrame(dataset_pool)
    export_path = os.path.join(OUTPUT_DIR, "Master_Training_Corpus.xlsx")
    master_df.to_excel(export_path, index=False)

    print(f"\n🎉 PHASE 3 SUCCESSFUL!")
    print(f"📊 Total valid rows extracted and aligned: {len(master_df)}")
    print(master_df['language_protocol'].value_counts())
    print(f"💾 Master compilation file saved to Drive at: {export_path}")
else:
    print("\n🚨 Pipeline Error: No usable text could be extracted. Verify that your PDFs contain text layers.")

📂 Found 15 PDF files in your storage directory.
⚙️ Processing documents through extraction layer...
[1/15] Parsing text layers from: DE_Sovereignty_Paper.pdf
      ✅ Text parsed successfully! Classified as: [de_DE]
------------------------------------------------------------
[2/15] Parsing text layers from: FR_Policy_AI_Sovereignty.pdf
      ✅ Text parsed successfully! Classified as: [en_XX]
------------------------------------------------------------
[3/15] Parsing text layers from: AR_Institutional_AI_Report.pdf
      ✅ Text parsed successfully! Classified as: [ar_AR]
------------------------------------------------------------
[4/15] Parsing text layers from: AR_Global_ITU_Trends.pdf
      ✅ Text parsed successfully! Classified as: [ar_AR]
------------------------------------------------------------
[5/15] Parsing text layers from: FR_Gov_Digital_Strategy.pdf
      ✅ Text parsed successfully! Classified as: [en_XX]
------------------------------------------------------------
[6/15] 

In [5]:
# =====================================================================
# 🧬 PHASE 1 - STEP 4: HUGGING FACE TOKENIZATION & FIELD PACKING
# =====================================================================
from datasets import Dataset
from transformers import AutoTokenizer

print("\n🧬 Initializing Hugging Face Tokenization Engine...")
BASE_MODEL = "facebook/mbart-large-50"

# Load tokenizer setup from mBART backbone
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Read the excel file we just exported above to ensure we are using the exact same data matrix
df_for_tensors = pd.read_excel(export_path)
df_for_tensors.dropna(subset=['document_body', 'target_summary'], inplace=True)

# Transition our Pandas frames directly into Arrow HF Dataset formats
hf_dataset = Dataset.from_pandas(df_for_tensors)

def encode_sequences(batch):
    """Encodes document text bodies and target labels into multi-dimensional matrix tensors."""
    # 1. Encode source text bodies safely up to model context windows
    model_inputs = tokenizer(
        batch["document_body"],
        max_length=1024,
        truncation=True,
        padding="max_length"
    )

    # 2. Tokenize baseline target summary labels
    labels = tokenizer(
        text_target=batch["target_summary"],
        max_length=250,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("⚡ Tokenizing text fields and exporting mathematical tensor maps...")
tokenized_dataset = hf_dataset.map(encode_sequences, batched=True)

# Cache tensor output folder directly to your Drive for immediate training initialization in Phase 2
tensor_save_path = os.path.join(OUTPUT_DIR, "tokenized_dataset_tensor")
tokenized_dataset.save_to_disk(tensor_save_path)

print(f"\n🚀 PHASE 1 IS NOW 100% COMPLETE!")
print(f"📁 Encoded model tensors safely preserved at: {tensor_save_path}")



🧬 Initializing Hugging Face Tokenization Engine...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

⚡ Tokenizing text fields and exporting mathematical tensor maps...


Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/15 [00:00<?, ? examples/s]


🚀 PHASE 1 IS NOW 100% COMPLETE!
📁 Encoded model tensors safely preserved at: /content/drive/MyDrive/New_Multilingual_AI_Project/processed_outputs/tokenized_dataset_tensor
Ready to be saved as '01_Data_Pipeline.ipynb' and pushed to GitHub!
